In [2]:
import pandas as pd
import numpy as np
from scipy import stats
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import platform, os

# ── 한글 폰트 ────────────────────────────────────────────────────
system = platform.system()
if system == "Windows":
    plt.rcParams['font.family'] = 'Malgun Gothic'
    font_prop = fm.FontProperties(family='Malgun Gothic')
elif system == "Darwin":
    plt.rcParams['font.family'] = 'AppleGothic'
    font_prop = fm.FontProperties(family='AppleGothic')
else:
    font_path = '/usr/share/fonts/truetype/nanum/NanumBarunGothicBold.ttf'
    if os.path.exists(font_path):
        fm.fontManager.addfont(font_path)
        font_prop = fm.FontProperties(fname=font_path)
        plt.rcParams['font.family'] = font_prop.get_name()
    else:
        font_prop = fm.FontProperties()
plt.rcParams['axes.unicode_minus'] = False

# ── 1. 데이터 로드 ───────────────────────────────────────────────
macro = pd.read_csv(r"C:\유비온프로젝트2\corporate-bankruptcy\영경\부실과 거시지표 관계\거시지표_통합.csv", encoding="utf-8-sig")
bad   = pd.read_csv(r"C:\유비온프로젝트2\corporate-bankruptcy\영경\부실과 거시지표 관계\연도별_부실비율.csv", encoding="utf-8-sig")
bad   = bad[["연도", "부실비율(%)"]].rename(columns={"부실비율(%)": "부실비율"})
bad   = bad[~bad["연도"].isin([2011, 2012])]  # 이상치 제거

# ── 2. 컬럼명 정리 ───────────────────────────────────────────────
macro = macro.rename(columns={
    "한국은행 기준금리":           "기준금리",
    "회사채(3년, BBB-)":          "회사채BBB",
    "국내총생산(명목, 원화표시)":   "GDP",
    "산업별대출금":                "대출금",
})

# ── 3. 지표 변환 ─────────────────────────────────────────────────
# 원값 그대로
# 기준금리, 실업률, 고용률 → 그대로 사용

# 스프레드 = 회사채BBB - 기준금리
macro["회사채스프레드"] = macro["회사채BBB"] - macro["기준금리"]

# YoY 증가율 (전년 대비 %)
for col, new_col in [
    ("생산자물가지수", "생산자물가_YoY"),
    ("소비자물가지수", "소비자물가_YoY"),
    ("GDP",           "GDP성장률_YoY"),
    ("대출금",         "대출증가율_YoY"),
]:
    macro[new_col] = macro[col].pct_change() * 100

# ── 4. 분석에 사용할 컬럼 선택 ──────────────────────────────────
use_cols = [
    "연도",
    "기준금리",
    "회사채스프레드",
    "실업률",
    "고용률",
    "생산자물가_YoY",
    "소비자물가_YoY",
    "GDP성장률_YoY",
    "대출증가율_YoY",
]
macro_clean = macro[use_cols].copy()

# ── 5. 부실비율과 병합 ───────────────────────────────────────────
merged = pd.merge(bad, macro_clean, on="연도").dropna()
var_list = [c for c in merged.columns if c not in ["연도", "부실비율"]]

print("=" * 72)
print("분석 데이터  (2011·2012 제외)")
print("=" * 72)
print(merged.to_string(index=False))

# ── 6. Spearman 상관분석 ─────────────────────────────────────────
print("\n" + "=" * 72)
print("Spearman 상관분석 결과")
print("=" * 72)
print(f"{'변수':<22} {'상관계수(ρ)':>12} {'p-value':>12} {'유의성':>8}")
print("-" * 72)

corr_results = {}
for var in var_list:
    r, p = stats.spearmanr(merged[var], merged["부실비율"])
    sig = "***" if p < 0.001 else "**" if p < 0.01 else "*" if p < 0.05 else "n.s."
    print(f"{var:<22} {r:>12.4f} {p:>12.4f} {sig:>8}")
    corr_results[var] = (r, p, sig)

print("\n* p<0.05  ** p<0.01  *** p<0.001  n.s. = 유의하지 않음")

# ── 7. 시각화 (산점도 + 회귀선) ──────────────────────────────────
n = len(var_list)
cols = 4
rows = (n + cols - 1) // cols
palette = ["#2196F3","#FF9800","#4CAF50","#E91E63",
           "#9C27B0","#00BCD4","#FF5722","#607D8B"]

fig, axes = plt.subplots(rows, cols, figsize=(20, rows * 5))
fig.patch.set_facecolor("#f8f9fa")
fig.suptitle("거시지표 vs 부실비율  —  Spearman 상관분석",
             fontproperties=font_prop, fontsize=14, fontweight="bold", y=1.02)

axes_flat = axes.flatten()
for i, var in enumerate(var_list):
    r, p, sig = corr_results[var]
    color = palette[i % len(palette)]
    ax = axes_flat[i]

    x, y = merged[var], merged["부실비율"]
    ax.scatter(x, y, color=color, s=70, zorder=3, edgecolors="white", linewidth=0.8)

    # 회귀선
    m, b = np.polyfit(x, y, 1)
    xl = np.linspace(x.min(), x.max(), 100)
    ax.plot(xl, m * xl + b, color=color, linewidth=2, alpha=0.7)

    # 연도 라벨
    for _, row in merged.iterrows():
        ax.annotate(str(int(row["연도"])), xy=(row[var], row["부실비율"]),
                    fontsize=6.5, ha="left", va="bottom",
                    fontproperties=font_prop, color="gray")

    ax.set_facecolor("#ffffff")
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.set_xlabel(var, fontproperties=font_prop, fontsize=9)
    ax.set_ylabel("부실비율 (%)", fontproperties=font_prop, fontsize=9)
    ax.set_title(f"{var}\nρ={r:.3f},  p={p:.3f}  ({sig})",
                 fontproperties=font_prop, fontsize=10, fontweight="bold")
    ax.grid(axis="both", linestyle="--", alpha=0.3)

# 빈 칸 숨기기
for i in range(n, len(axes_flat)):
    axes_flat[i].set_visible(False)

plt.tight_layout(pad=2.5)
plt.savefig("spearman_chart.png", dpi=150, bbox_inches="tight")
print("\n그래프 저장 완료 → spearman_chart.png")

분석 데이터  (2011·2012 제외)
  연도  부실비율     기준금리  회사채스프레드      실업률       고용률  생산자물가_YoY  소비자물가_YoY  GDP성장률_YoY  대출증가율_YoY
2013  5.06 2.583333 6.252833 3.100000 59.808333  -1.601949   1.301348    4.403884   4.849775
2014  5.04 2.333333 6.378833 3.491667 60.508333  -0.532020   1.274774    4.299762   6.650899
2015  4.43 1.645833 6.354667 3.591667 60.541667  -4.014373   0.706332    6.243036   6.851093
2016  3.81 1.354167 6.571750 3.675000 60.558333  -1.815091   0.971686    5.299395   4.469299
2017  4.09 1.291667 7.264583 3.683333 60.841667   3.450688   1.944332    5.521322   6.700721
2018  4.26 1.541667 7.275833 3.833333 60.716667   1.877430   1.475839    3.760693   6.656975
2019  4.90 1.562500 6.554667 3.783333 60.958333   0.022403   0.383000    1.675148   7.732073
2020  5.16 0.666667 7.741250 3.941667 60.075000  -0.454591   0.537288    0.875833  15.421685
2021  4.83 0.645833 7.684667 3.675000 60.483333   6.377500   2.498333    7.940202  13.461670
2022  5.21 2.125000 7.863250 2.883333 62.058333

## 1년 시차

In [3]:
import pandas as pd
import numpy as np
from scipy import stats
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import platform, os

# ── 한글 폰트 ────────────────────────────────────────────────────
system = platform.system()
if system == "Windows":
    plt.rcParams['font.family'] = 'Malgun Gothic'
    font_prop = fm.FontProperties(family='Malgun Gothic')
elif system == "Darwin":
    plt.rcParams['font.family'] = 'AppleGothic'
    font_prop = fm.FontProperties(family='AppleGothic')
else:
    font_path = '/usr/share/fonts/truetype/nanum/NanumBarunGothicBold.ttf'
    if os.path.exists(font_path):
        fm.fontManager.addfont(font_path)
        font_prop = fm.FontProperties(fname=font_path)
        plt.rcParams['font.family'] = font_prop.get_name()
    else:
        font_prop = fm.FontProperties()
plt.rcParams['axes.unicode_minus'] = False

# ── 1. 데이터 로드 ───────────────────────────────────────────────
macro = pd.read_csv(r"C:\유비온프로젝트2\corporate-bankruptcy\영경\부실과 거시지표 관계\거시지표_통합.csv", encoding="utf-8-sig")
bad   = pd.read_csv(r"C:\유비온프로젝트2\corporate-bankruptcy\영경\부실과 거시지표 관계\연도별_부실비율.csv", encoding="utf-8-sig")
bad   = bad[["연도", "부실비율(%)"]].rename(columns={"부실비율(%)": "부실비율"})
bad   = bad[bad["연도"] >= 2012]   # 2012년부터 포함 (시차 적용 시 2013부터 매칭)

# ── 2. 컬럼명 정리 ───────────────────────────────────────────────
macro = macro.rename(columns={
    "한국은행 기준금리":           "기준금리",
    "회사채(3년, BBB-)":          "회사채BBB",
    "국내총생산(명목, 원화표시)":   "GDP",
    "산업별대출금":                "대출금",
})

# ── 3. 지표 변환 ─────────────────────────────────────────────────
macro["회사채스프레드"] = macro["회사채BBB"] - macro["기준금리"]

for col, new_col in [
    ("생산자물가지수", "생산자물가_YoY"),
    ("소비자물가지수", "소비자물가_YoY"),
    ("GDP",           "GDP성장률_YoY"),
    ("대출금",         "대출증가율_YoY"),
]:
    macro[new_col] = macro[col].pct_change() * 100

# ── 4. 분석 컬럼 선택 ────────────────────────────────────────────
use_cols = [
    "연도",
    "기준금리",
    "회사채스프레드",
    "실업률",
    "고용률",
    "생산자물가_YoY",
    "소비자물가_YoY",
    "GDP성장률_YoY",
    "대출증가율_YoY",
]
macro_clean = macro[use_cols].copy()

# ── 5. 1년 시차 적용: 거시지표 연도 + 1 = 부실비율 연도 ──────────
macro_clean = macro_clean.copy()
macro_clean["연도"] = macro_clean["연도"] + 1

# ── 6. 병합 ─────────────────────────────────────────────────────
merged = pd.merge(bad, macro_clean, on="연도").dropna()
var_list = [c for c in merged.columns if c not in ["연도", "부실비율"]]

print("=" * 80)
print("분석 데이터  (거시지표: 전년도 기준 / 부실비율: 당해연도 기준 — 1년 시차)")
print("=" * 80)
print(merged.to_string(index=False))

# ── 7. Spearman 상관분석 ─────────────────────────────────────────
print("\n" + "=" * 72)
print("Spearman 상관분석 결과  (1년 시차)")
print("=" * 72)
print(f"{'변수':<22} {'상관계수(ρ)':>12} {'p-value':>12} {'유의성':>8}")
print("-" * 72)

corr_results = {}
for var in var_list:
    r, p = stats.spearmanr(merged[var], merged["부실비율"])
    sig = "***" if p < 0.001 else "**" if p < 0.01 else "*" if p < 0.05 else "n.s."
    print(f"{var:<22} {r:>12.4f} {p:>12.4f} {sig:>8}")
    corr_results[var] = (r, p, sig)

print("\n* p<0.05  ** p<0.01  *** p<0.001  n.s. = 유의하지 않음")

# ── 8. 시각화 ────────────────────────────────────────────────────
n = len(var_list)
cols = 4
rows = (n + cols - 1) // cols
palette = ["#2196F3","#FF9800","#4CAF50","#E91E63",
           "#9C27B0","#00BCD4","#FF5722","#607D8B"]

fig, axes = plt.subplots(rows, cols, figsize=(20, rows * 5))
fig.patch.set_facecolor("#f8f9fa")
fig.suptitle("거시지표(전년도) vs 부실비율(당해연도)  —  Spearman 상관분석  [1년 시차]",
             fontproperties=font_prop, fontsize=13, fontweight="bold", y=1.02)

axes_flat = axes.flatten()
for i, var in enumerate(var_list):
    r, p, sig = corr_results[var]
    color = palette[i % len(palette)]
    ax = axes_flat[i]

    x, y = merged[var], merged["부실비율"]
    ax.scatter(x, y, color=color, s=70, zorder=3, edgecolors="white", linewidth=0.8)

    m, b = np.polyfit(x, y, 1)
    xl = np.linspace(x.min(), x.max(), 100)
    ax.plot(xl, m * xl + b, color=color, linewidth=2, alpha=0.7)

    for _, row in merged.iterrows():
        ax.annotate(str(int(row["연도"])), xy=(row[var], row["부실비율"]),
                    fontsize=6.5, ha="left", va="bottom",
                    fontproperties=font_prop, color="gray")

    ax.set_facecolor("#ffffff")
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.set_xlabel(var, fontproperties=font_prop, fontsize=9)
    ax.set_ylabel("부실비율 (%)", fontproperties=font_prop, fontsize=9)
    ax.set_title(f"{var}\nρ={r:.3f},  p={p:.3f}  ({sig})",
                 fontproperties=font_prop, fontsize=10, fontweight="bold")
    ax.grid(axis="both", linestyle="--", alpha=0.3)

for i in range(n, len(axes_flat)):
    axes_flat[i].set_visible(False)

plt.tight_layout(pad=2.5)
plt.savefig("spearman_lag1_chart.png", dpi=150, bbox_inches="tight")
print("\n그래프 저장 완료 → spearman_lag1_chart.png")

분석 데이터  (거시지표: 전년도 기준 / 부실비율: 당해연도 기준 — 1년 시차)
  연도  부실비율     기준금리  회사채스프레드      실업률       고용률  생산자물가_YoY  소비자물가_YoY  GDP성장률_YoY  대출증가율_YoY
2013  5.06 3.062500 6.275833 3.225000 59.608333   0.693632   2.187071    3.872666   2.629374
2014  5.04 2.583333 6.252833 3.100000 59.808333  -1.601949   1.301348    4.403884   4.849775
2015  4.43 2.333333 6.378833 3.491667 60.508333  -0.532020   1.274774    4.299762   6.650899
2016  3.81 1.645833 6.354667 3.591667 60.541667  -4.014373   0.706332    6.243036   6.851093
2017  4.09 1.354167 6.571750 3.675000 60.558333  -1.815091   0.971686    5.299395   4.469299
2018  4.26 1.291667 7.264583 3.683333 60.841667   3.450688   1.944332    5.521322   6.700721
2019  4.90 1.541667 7.275833 3.833333 60.716667   1.877430   1.475839    3.760693   6.656975
2020  5.16 1.562500 6.554667 3.783333 60.958333   0.022403   0.383000    1.675148   7.732073
2021  4.83 0.666667 7.741250 3.941667 60.075000  -0.454591   0.537288    0.875833  15.421685
2022  5.21 0.645833 7.6